In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from experta import *

import sys
import os
sys.path.append(os.path.abspath(".."))
from rule_based_system.Rules import HeartExpert

In [2]:
ml_df = pd.read_csv("../data/cleaned_data_for_ml.csv")
expert_df = pd.read_csv("../data/cleaned_data_for_expert.csv")

In [3]:
X_ml = ml_df.drop('target', axis=1)
y_ml = ml_df['target']

X_train_ml, X_test_ml, y_train_ml, y_test_ml = train_test_split(
    X_ml, y_ml, test_size=0.2, random_state=42
)

In [4]:
X_exp = expert_df.drop('target', axis=1)
y_exp = expert_df['target']

X_train_exp, X_test_exp, y_train_exp, y_test_exp = train_test_split(
    X_exp, y_exp, test_size=0.2, random_state=42
)

In [5]:
model = joblib.load("../ml_model/decision_tree_model.pkl")

y_pred_ml = model.predict(X_test_ml)

dt_results = {
    "Accuracy": accuracy_score(y_test_ml, y_pred_ml),
    "Precision": precision_score(y_test_ml, y_pred_ml),
    "Recall": recall_score(y_test_ml, y_pred_ml),
    "F1": f1_score(y_test_ml, y_pred_ml)
}

print("Decision Tree Results:")
print(dt_results)

Decision Tree Results:
{'Accuracy': 0.7358490566037735, 'Precision': 0.8387096774193549, 'Recall': 0.7428571428571429, 'F1': 0.7878787878787878}


In [6]:
expert_preds = []

for i in range(len(X_test_exp)):
    engine = HeartExpert()
    engine.reset()

    row = X_test_exp.iloc[i]

    for col in X_test_exp.columns:
        engine.declare(Fact(**{col: row[col]}))

    engine.run()

    expert_preds.append(engine.get_result())

In [7]:
expert_results = {
    "Accuracy": accuracy_score(y_test_exp, expert_preds),
    "Precision": precision_score(y_test_exp, expert_preds),
    "Recall": recall_score(y_test_exp, expert_preds),
    "F1": f1_score(y_test_exp, expert_preds)
}

print("Expert System Results:")
print(expert_results)

Expert System Results:
{'Accuracy': 0.37735849056603776, 'Precision': 0.5416666666666666, 'Recall': 0.37142857142857144, 'F1': 0.4406779661016949}


In [8]:
comparison = pd.DataFrame({
    "Decision Tree": dt_results,
    "Expert System": expert_results
})

print("\n=== Final Comparison ===")
print(comparison)


=== Final Comparison ===
           Decision Tree  Expert System
Accuracy        0.735849       0.377358
Precision       0.838710       0.541667
Recall          0.742857       0.371429
F1              0.787879       0.440678


# Comparison Summary
The Decision Tree model achieved higher performance because it learns patterns from data.
The Expert System uses rule-based reasoning with a scoring mechanism.
The Expert System is more interpretable and explainable.
The Decision Tree is more accurate and scalable.

# Conclusion
Decision Tree is better for prediction accuracy.
Expert System is better for explainability.